# Area Classifier — SMOTE Oversampling

Questo notebook esplora l'uso di tecniche **SMOTE** (Synthetic Minority Oversampling Technique)
per migliorare la classificazione dell'area funzionale dei ticket.

**Baseline (AreaClassifier v2)**
- LinearSVC con `class_weight='balanced'`
- Macro F1 test: **0.6769** | Accuracy: **0.7677**

**Obiettivo**: verificare se bilanciare il training set via oversampling sintetico
(invece di pesare le classi sul loss) porta a performance migliori,
in particolare sulle classi minoritarie problematiche:
- `rendicontazione_flussi`
- `protocollo_documentale_delibere`
- `area_sistemistica` (sovra-predetta)

**Varianti testate**:
1. Baseline → `LinearSVC(class_weight='balanced')`  *(riferimento)*
2. SMOTE → oversample minority classes → `LinearSVC(class_weight=None)`
3. BorderlineSMOTE → oversampling solo sui campioni di confine (più selettivo)
4. SMOTEENN → SMOTE + Edited Nearest Neighbours (oversample + cleaning rumore)

In [ ]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings, time, json
from pathlib import Path

from sklearn.svm import LinearSVC
from sklearn.preprocessing import OrdinalEncoder, LabelEncoder
from sklearn.metrics import (
    classification_report, f1_score, accuracy_score, confusion_matrix
)
from sklearn.preprocessing import normalize

from imblearn.over_sampling import SMOTE, BorderlineSMOTE
from imblearn.combine import SMOTEENN

warnings.filterwarnings('ignore')
print("Import completati.")

In [ ]:
# ── Percorsi ────────────────────────────────────────────────────────────────
ROOT         = Path("..").resolve()
DATA_DIR     = ROOT / "data"
EMB_DIR      = ROOT / "embeddings"
MODELLI_DIR  = ROOT / "modelli" / "area_v3_smote"
MODELLI_DIR.mkdir(parents=True, exist_ok=True)

DATASET_PATH  = DATA_DIR / "dataset_clean.csv"
EMB_TRAIN     = EMB_DIR  / "e5_train.npy"
EMB_TEST      = EMB_DIR  / "e5_test.npy"

# ── Parametri ────────────────────────────────────────────────────────────────
SOGLIA_SPLIT  = "2025-11-01"
TARGET_COL    = "area_v2"
TESTO_COL     = "testo_input"

KW_COLS = [
    "kw_s381_codice", "kw_s381_rapportino", "kw_s381_timbrate",
    "kw_s381_calendario_presenze",
    "kw_ter_unodomo", "kw_ter_distretto",
    "kw_san_terapia", "kw_san_pai", "kw_san_css", "kw_san_diario",
    "kw_san_contenzioni", "kw_san_farmaco",
    "kw_pas_iva", "kw_pas_cespiti", "kw_pas_prima_nota",
    "kw_pas_ammortamento", "kw_pas_analitica", "kw_pas_reverse",
    "kw_pas_fornitore",
    "kw_att_retta", "kw_att_pagopa", "kw_att_sdd",
    "kw_att_portale_utenti", "kw_att_fattura_elettronica",
    "kw_sis_installazione_programma",
]

CAT_COL = "priorita_iniziale_cliente"

print("Configurazione OK")
print(f"  Dataset  : {DATASET_PATH}")
print(f"  Emb train: {EMB_TRAIN}")
print(f"  Emb test : {EMB_TEST}")
print(f"  Output   : {MODELLI_DIR}")

## 1 — Caricamento dataset e split temporale

In [ ]:
df = pd.read_csv(DATASET_PATH, parse_dates=["data_creazione"])
print(f"Dataset caricato: {df.shape[0]:,} righe × {df.shape[1]} colonne")

# Teniamo solo le righe con area_v2 valorizzata
df_area = df.dropna(subset=[TARGET_COL]).copy()
print(f"Con area_v2 valorizzata: {df_area.shape[0]:,} righe")
print(f"Classi ({df_area[TARGET_COL].nunique()}): {sorted(df_area[TARGET_COL].unique())}")

In [ ]:
# Split temporale identico a AreaClassifier v2
mask_train = df_area["data_creazione"] < SOGLIA_SPLIT
mask_test  = df_area["data_creazione"] >= SOGLIA_SPLIT

df_train = df_area[mask_train].copy()
df_test  = df_area[mask_test].copy()

print(f"Train (< {SOGLIA_SPLIT}): {len(df_train):,} ticket")
print(f"Test  (>= {SOGLIA_SPLIT}): {len(df_test):,} ticket")

## 2 — Distribuzione classi nel training set (sbilanciamento)

In [ ]:
counts = df_train[TARGET_COL].value_counts().sort_values()
pct    = (counts / counts.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(counts.index, counts.values, color="steelblue", edgecolor="white")
for bar, n, p in zip(bars, counts.values, pct.values):
    ax.text(bar.get_width() + 30, bar.get_y() + bar.get_height() / 2,
            f"{n:,}  ({p}%)", va="center", fontsize=9)
ax.set_xlabel("Ticket nel training set")
ax.set_title("Distribuzione classi — Training set (split temporale)")
ax.set_xlim(0, counts.max() * 1.25)
plt.tight_layout()
plt.show()

ratio = counts.max() / counts.min()
print(f"\nRapporto max/min classi: {ratio:.1f}x  "
      f"(max={counts.idxmax()}: {counts.max():,}, "
      f"min={counts.idxmin()}: {counts.min():,})")

## 3 — Costruzione feature matrix

In [ ]:
# ── Caricamento embedding pre-calcolati ──────────────────────────────────────
# Gli embedding coprono TUTTO il dataset (train priority + test priority).
# Dobbiamo allinearli agli indici del sottoinsieme area.

# Il dataset_clean.csv è ordinato per data_creazione; gli embedding
# sono stati calcolati sullo stesso ordine con split priority
# (soglia 2025-11-01 → 46312 train, 12032 test).
# Usiamo get_indexer per mappare gli indici area → embedding.

df_all_train = df[df["data_creazione"] < SOGLIA_SPLIT].copy()
df_all_test  = df[df["data_creazione"] >= SOGLIA_SPLIT].copy()

X_emb_train_full = np.load(EMB_TRAIN)   # (N_train_priority, 768)
X_emb_test_full  = np.load(EMB_TEST)    # (N_test_priority,  768)

print(f"Embedding train full shape: {X_emb_train_full.shape}")
print(f"Embedding test  full shape: {X_emb_test_full.shape}")
print(f"df_all_train rows         : {len(df_all_train)}")
print(f"df_all_test  rows         : {len(df_all_test)}")

In [ ]:
# Allineamento indici area → embedding
idx_train = df_all_train.index.get_indexer(df_train.index)
idx_test  = df_all_test.index.get_indexer(df_test.index)

assert (idx_train < 0).sum() == 0, "Indici train non trovati negli embedding!"
assert (idx_test  < 0).sum() == 0, "Indici test  non trovati negli embedding!"

X_emb_train = X_emb_train_full[idx_train]   # (n_train_area, 768)
X_emb_test  = X_emb_test_full[idx_test]     # (n_test_area,  768)

# L2-normalizzazione (già normalizzati dal compute_embeddings, ma ripetiamo per sicurezza)
X_emb_train = normalize(X_emb_train, norm="l2")
X_emb_test  = normalize(X_emb_test,  norm="l2")

print(f"X_emb_train: {X_emb_train.shape}")
print(f"X_emb_test : {X_emb_test.shape}")

In [ ]:
# ── Feature categoriali: OHE priorita_iniziale_cliente ───────────────────────
enc = OrdinalEncoder(
    categories=[["P1", "P2", "P3", "P4"]],
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    encoded_missing_value=-1,
)
enc.fit(df_train[[CAT_COL]])

def make_cat_ohe(df_split, encoder):
    codes = encoder.transform(df_split[[CAT_COL]]).astype(int).flatten()
    n_cats = len(encoder.categories_[0])
    ohe = np.zeros((len(df_split), n_cats + 1), dtype=np.float32)  # +1 per NaN/unknown
    for i, c in enumerate(codes):
        ohe[i, c] = 1.0   # c==-1 → colonna indice 0 che non corrisponde a nessuna priorità
    return ohe

X_cat_train = make_cat_ohe(df_train, enc)
X_cat_test  = make_cat_ohe(df_test,  enc)

# ── Feature keyword booleane ──────────────────────────────────────────────────
kw_cols_present = [c for c in KW_COLS if c in df_train.columns]
X_kw_train = df_train[kw_cols_present].fillna(0).values.astype(np.float32)
X_kw_test  = df_test[kw_cols_present].fillna(0).values.astype(np.float32)

print(f"Feature categoriali  : {X_cat_train.shape[1]}d")
print(f"Feature keyword      : {X_kw_train.shape[1]}d")
print(f"Feature embedding    : {X_emb_train.shape[1]}d")
print(f"Totale feature       : {X_emb_train.shape[1] + X_cat_train.shape[1] + X_kw_train.shape[1]}d")

In [ ]:
# ── Concatenazione feature (dense) ───────────────────────────────────────────
# SMOTE richiede array denso → usiamo np.hstack.
# Dimensioni indicative: 41k × 797 float32 ≈ 130 MB → gestibile.

X_train = np.hstack([X_emb_train, X_cat_train, X_kw_train]).astype(np.float32)
X_test  = np.hstack([X_emb_test,  X_cat_test,  X_kw_test]).astype(np.float32)

y_train = df_train[TARGET_COL].values
y_test  = df_test[TARGET_COL].values

print(f"X_train: {X_train.shape}  ({X_train.nbytes / 1e6:.1f} MB)")
print(f"X_test : {X_test.shape}  ({X_test.nbytes / 1e6:.1f} MB)")
print(f"y_train classi: {np.unique(y_train)}")

## 4 — Applicazione SMOTE

### Strategia di oversampling

Per evitare di gonfiare inutilmente le classi maggioritarie, usiamo
`sampling_strategy='not majority'` che porta tutte le classi al livello della
seconda più grande (non alla più grande). Questo è più conservativo e riduce
il rischio di overfitting sui campioni sintetici.

In [ ]:
# ── SMOTE standard ───────────────────────────────────────────────────────────
print("Applicazione SMOTE standard (k_neighbors=5)...")
t0 = time.time()
smote = SMOTE(
    sampling_strategy="not majority",
    k_neighbors=5,
    random_state=42,
    n_jobs=-1,
)
X_smote, y_smote = smote.fit_resample(X_train, y_train)
print(f"  Tempo: {time.time()-t0:.1f}s")
print(f"  Prima: {X_train.shape[0]:,} campioni")
print(f"  Dopo : {X_smote.shape[0]:,} campioni")

dist_smote = pd.Series(y_smote).value_counts().sort_index()
print("\nDistribuzione dopo SMOTE:")
print(dist_smote.to_string())

In [ ]:
# ── BorderlineSMOTE ───────────────────────────────────────────────────────────
# Oversampling solo sui campioni vicini al confine decisionale → più selettivo
print("Applicazione BorderlineSMOTE (kind='borderline-1')...")
t0 = time.time()
bl_smote = BorderlineSMOTE(
    sampling_strategy="not majority",
    k_neighbors=5,
    m_neighbors=10,
    kind="borderline-1",
    random_state=42,
    n_jobs=-1,
)
X_blsmote, y_blsmote = bl_smote.fit_resample(X_train, y_train)
print(f"  Tempo: {time.time()-t0:.1f}s")
print(f"  Campioni dopo: {X_blsmote.shape[0]:,}")

In [ ]:
# ── SMOTEENN ─────────────────────────────────────────────────────────────────
# SMOTE + Edited Nearest Neighbours: oversampla le minority e poi rimuove
# campioni rumorosi vicini al confine (sia maggioritari che sintetici).
print("Applicazione SMOTEENN...")
t0 = time.time()
smoteenn = SMOTEENN(
    sampling_strategy="not majority",
    random_state=42,
    n_jobs=-1,
)
X_smoteenn, y_smoteenn = smoteenn.fit_resample(X_train, y_train)
print(f"  Tempo: {time.time()-t0:.1f}s")
print(f"  Campioni dopo: {X_smoteenn.shape[0]:,}")

dist_enn = pd.Series(y_smoteenn).value_counts().sort_index()
print("\nDistribuzione dopo SMOTEENN:")
print(dist_enn.to_string())

## 5 — Training dei modelli

In [ ]:
def train_eval(X_tr, y_tr, X_te, y_te, C=10.0, class_weight=None, label=""):
    """Allena LinearSVC e restituisce metriche + predizioni."""
    clf = LinearSVC(C=C, class_weight=class_weight, max_iter=5000, random_state=42)
    t0 = time.time()
    clf.fit(X_tr, y_tr)
    elapsed = time.time() - t0
    y_pred = clf.predict(X_te)
    macro_f1  = f1_score(y_te, y_pred, average="macro")
    accuracy  = accuracy_score(y_te, y_pred)
    report    = classification_report(y_te, y_pred, output_dict=True, zero_division=0)
    print(f"[{label}]  Macro F1: {macro_f1:.4f}  |  Accuracy: {accuracy:.4f}  |  Train: {elapsed:.1f}s")
    return clf, y_pred, macro_f1, accuracy, report

# ── 0. Baseline: class_weight='balanced' (stesso di v2) ──────────────────────
clf_base, y_pred_base, f1_base, acc_base, rpt_base = train_eval(
    X_train, y_train, X_test, y_test,
    C=10.0, class_weight="balanced",
    label="Baseline  class_weight=balanced"
)

# ── 1. SMOTE → no class_weight ────────────────────────────────────────────────
clf_smote, y_pred_smote, f1_smote, acc_smote, rpt_smote = train_eval(
    X_smote, y_smote, X_test, y_test,
    C=10.0, class_weight=None,
    label="SMOTE     class_weight=None    "
)

# ── 2. BorderlineSMOTE → no class_weight ─────────────────────────────────────
clf_blsmote, y_pred_blsmote, f1_blsmote, acc_blsmote, rpt_blsmote = train_eval(
    X_blsmote, y_blsmote, X_test, y_test,
    C=10.0, class_weight=None,
    label="BL-SMOTE  class_weight=None    "
)

# ── 3. SMOTEENN → no class_weight ────────────────────────────────────────────
clf_smoteenn, y_pred_smoteenn, f1_smoteenn, acc_smoteenn, rpt_smoteenn = train_eval(
    X_smoteenn, y_smoteenn, X_test, y_test,
    C=10.0, class_weight=None,
    label="SMOTEENN  class_weight=None    "
)

## 6 — Confronto performance

In [ ]:
# ── Tabella riepilogativa globale ─────────────────────────────────────────────
results_global = pd.DataFrame([
    {"Modello": "Baseline (class_weight=balanced)", "Macro F1": f1_base,     "Accuracy": acc_base},
    {"Modello": "SMOTE",                            "Macro F1": f1_smote,    "Accuracy": acc_smote},
    {"Modello": "BorderlineSMOTE",                  "Macro F1": f1_blsmote,  "Accuracy": acc_blsmote},
    {"Modello": "SMOTEENN",                         "Macro F1": f1_smoteenn, "Accuracy": acc_smoteenn},
]).set_index("Modello")

results_global["Delta F1 vs Baseline"] = results_global["Macro F1"] - f1_base
results_global = results_global.round(4)

print(results_global.to_string())

# Barplot confronto Macro F1
fig, ax = plt.subplots(figsize=(8, 3.5))
colors = ["#4C72B0", "#DD8452", "#55A868", "#C44E52"]
bars = ax.barh(results_global.index, results_global["Macro F1"], color=colors, edgecolor="white")
for bar, v in zip(bars, results_global["Macro F1"]):
    ax.text(bar.get_width() + 0.002, bar.get_y() + bar.get_height()/2,
            f"{v:.4f}", va="center", fontsize=10)
ax.axvline(f1_base, color="gray", linestyle="--", lw=1, label="Baseline")
ax.set_xlabel("Macro F1 (test set)")
ax.set_title("Confronto Macro F1 — Area Classifier SMOTE")
ax.set_xlim(0, 1)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── F1 per classe — tutti i modelli a confronto ───────────────────────────────
classi = sorted(set(y_test))

def extract_f1_per_class(report, classi):
    return {c: report.get(c, {}).get("f1-score", 0.0) for c in classi}

f1_per_class = pd.DataFrame({
    "Baseline":        extract_f1_per_class(rpt_base,     classi),
    "SMOTE":           extract_f1_per_class(rpt_smote,    classi),
    "BorderlineSMOTE": extract_f1_per_class(rpt_blsmote,  classi),
    "SMOTEENN":        extract_f1_per_class(rpt_smoteenn, classi),
}).T

fig, ax = plt.subplots(figsize=(12, 4))
x = np.arange(len(classi))
w = 0.2
for i, (modello, color) in enumerate(zip(f1_per_class.index, colors)):
    ax.bar(x + i*w, f1_per_class.loc[modello], w, label=modello, color=color, edgecolor="white")
ax.set_xticks(x + 1.5*w)
ax.set_xticklabels(classi, rotation=30, ha="right", fontsize=9)
ax.set_ylabel("F1-score")
ax.set_ylim(0, 1)
ax.set_title("F1 per classe — confronto modelli")
ax.legend(fontsize=8)
ax.axhline(0.5, color="red", lw=0.8, linestyle=":")
plt.tight_layout()
plt.show()

print("\nF1 per classe (tabella):")
print(f1_per_class.round(3).to_string())

## 7 — Report dettagliato del miglior modello

In [ ]:
# Identifica il miglior modello per Macro F1
scores = {
    "Baseline":        (f1_base,     y_pred_base,     rpt_base),
    "SMOTE":           (f1_smote,    y_pred_smote,    rpt_smote),
    "BorderlineSMOTE": (f1_blsmote,  y_pred_blsmote,  rpt_blsmote),
    "SMOTEENN":        (f1_smoteenn, y_pred_smoteenn, rpt_smoteenn),
}
best_name = max(scores, key=lambda k: scores[k][0])
best_f1, best_pred, best_rpt = scores[best_name]

print(f"=== Miglior modello: {best_name} (Macro F1 = {best_f1:.4f}) ===\n")
print(classification_report(y_test, best_pred, zero_division=0))

In [ ]:
# ── Confusion matrix del miglior modello ─────────────────────────────────────
cm = confusion_matrix(y_test, best_pred, labels=classi, normalize="true")
fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    cm, annot=True, fmt=".2f",
    xticklabels=classi, yticklabels=classi,
    cmap="Blues", linewidths=0.3, ax=ax
)
ax.set_xlabel("Predetto")
ax.set_ylabel("Reale")
ax.set_title(f"Confusion Matrix normalizzata — {best_name}")
plt.xticks(rotation=35, ha="right", fontsize=8)
plt.yticks(rotation=0, fontsize=8)
plt.tight_layout()
plt.show()

## 8 — Salvataggio del miglior modello

In [ ]:
import pickle

# Mappa nome → (clf, oversampler_label, X_tr_usato, y_tr_usato)
clf_map = {
    "Baseline":        clf_base,
    "SMOTE":           clf_smote,
    "BorderlineSMOTE": clf_blsmote,
    "SMOTEENN":        clf_smoteenn,
}
best_clf = clf_map[best_name]

# Salva classificatore
with open(MODELLI_DIR / "classificatore_svc.pkl", "wb") as f:
    pickle.dump(best_clf, f)

# Salva encoder categoriale
with open(MODELLI_DIR / "encoder_cat.pkl", "wb") as f:
    pickle.dump(enc, f)

# Salva metadata
metadata = {
    "versione":             "v3_smote",
    "modello_embedding":    "intfloat/multilingual-e5-base",
    "prefisso_e5":          "query: ",
    "classificatore":       "LinearSVC",
    "C":                    10.0,
    "class_weight":         "balanced" if best_name == "Baseline" else None,
    "oversampling":         best_name,
    "feature": [
        "embedding_e5_768d",
        "ohe_priorita_iniziale_cliente",
        "keyword_booleane",
    ],
    "classi":               classi,
    "split":                "temporale",
    "soglia_split":         SOGLIA_SPLIT,
    "macro_f1_test":        round(best_f1, 4),
    "accuracy_test":        round(accuracy_score(y_test, best_pred), 4),
    "n_train_originale":    int(X_train.shape[0]),
    "n_test":               int(X_test.shape[0]),
    "keyword_cols":         kw_cols_present,
    "confronto_modelli": {
        k: round(v[0], 4) for k, v in scores.items()
    },
}

with open(MODELLI_DIR / "metadata.json", "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

print(f"Modello salvato in: {MODELLI_DIR}")
print(f"  Miglior variante: {best_name}")
print(f"  Macro F1 test   : {best_f1:.4f}")
print(json.dumps(metadata["confronto_modelli"], indent=2))